In [1]:
import sys, os
import json
import torch
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))

from spectral_detection.analysis.pca_pipeline import pipeline

We first load the pipeline that we will be using

In [2]:
pipe = pipeline(L=21, H=32, K=10)

Let us test on a couple of individual data sets first.

In [ ]:
# X1, ids1 = pipe.load_pt(r"../data/spectral/truthfulQA_Judgelabels_and_eigs_top10.pt")

# X1_pca = pipe.fit_transform(X1)

# print("Original shape:", X1.shape)
# print("Reduced shape:", X1_pca.shape)

In [ ]:
# X2, ids2 = pipe.load_pt(r"../data/spectral/temp_1/triviaqa_t1.0_n1_eigen.pt", feature_key="eigs_top10")

# X2_pca = pipe.fit_transform(X2)

# print("Original shape:", X2.shape)
# print("Reduced shape:", X2_pca.shape)

## TriviaQA data

In [55]:
def build_training_dataset_eigen_only(jsonl_path: str, pt_path: str):

    with open(jsonl_path, "r") as f:
        df_meta = pd.DataFrame([json.loads(line) for line in f])

    pt_payload = torch.load(pt_path, map_location="cpu", weights_only=True)
    feature_dict = pt_payload.get("data", pt_payload)

    # Extract the eigenvalue arrays
    feature_rows = []
    for record_id, payload in list(feature_dict.items()):
        eig_array = payload.numpy().astype(np.float32)

        feature_rows.append({"id": record_id, "features": eig_array})

    df_features = pd.DataFrame(feature_rows)

    # Inner join to guarantee perfect alignment between features and labels
    df_final = pd.merge(df_meta, df_features, on="id", how="inner")

    # Filter errors and construct the binary label vector
    df_final = df_final[df_final["correctness"] != "error"]
    df_final["label"] = df_final["correctness"].apply(lambda x: 1 if x.lower() == "correct" else 0)

    # Formulate the X and y matrices
    X = np.vstack(df_final["features"].values)
    y = df_final["label"].values

    print(f"Feature Matrix (X) shape: {X.shape}")
    print(f"Label Vector (y) shape: {y.shape}")

    return df_final, X, y

In [56]:
# jsonl_pt = r"../data/spectral/temp_1/triviaqa_t1.0_n1.jsonl"
eigen_pt = r"../data/spectral/temp_1/triviaqa_Judgelabels_and_eigs_top10.pt"

payload = torch.load(eigen_pt, map_location="cpu")

# print(type(payload))
# print(payload.keys())

rows = []

for sample_id, item in payload["data"].items():

    eigvals = item["eig_top10"].numpy()

    row = {
        "id": sample_id,
        "label": item["label"],
        # "domain": item.get("domain", None),
    }

    # expand eigenvalues into columns
    for i, v in enumerate(eigvals):
        row[f"eig_{i}"] = float(v)

    rows.append(row)

df1 = pd.DataFrame(rows)

In [ ]:
df1["label"] = df1["label"].apply(lambda x: 1 if x.lower() == "incorrect" else 0)
df1.head()

,id,label,eig_0,eig_1,eig_2,eig_3,eig_4,eig_5,eig_6,eig_7,...,eig_6710,eig_6711,eig_6712,eig_6713,eig_6714,eig_6715,eig_6716,eig_6717,eig_6718,eig_6719
0,triviaqa_00000_t1.0_ans00,0,0.353516,0.104980,0.081055,0.072754,0.053711,0.046875,0.038818,0.032227,...,0.005249,0.001221,0.000977,0.000000,-0.000557,-0.002380,-0.002502,-0.003708,-0.004761,-0.005432
1,triviaqa_00001_t1.0_ans00,0,0.261719,0.112793,0.056641,0.050781,0.027832,0.025879,0.024536,0.020752,...,0.042969,0.011902,0.001709,0.001404,0.000000,-0.000305,-0.001587,-0.002716,-0.002945,-0.003052
2,triviaqa_00002_t1.0_ans00,1,0.283203,0.201172,0.089844,0.088867,0.067871,0.065430,0.054199,0.042725,...,0.030273,0.005493,0.004761,0.000366,0.000000,-0.000664,-0.000671,-0.001297,-0.003204,-0.003265
3,triviaqa_00003_t1.0_ans00,0,0.178711,0.142578,0.044434,0.039307,0.033203,0.030518,0.029175,0.025757,...,0.012085,0.002869,0.002808,0.001495,0.000916,0.000000,-0.000366,-0.001984,-0.002289,-0.003143
4,triviaqa_00004_t1.0_ans00,0,0.127930,0.073730,0.064941,0.052002,0.038330,0.037598,0.026855,0.023438,...,0.007385,0.006104,0.005157,0.003326,0.002197,0.000000,-0.000519,-0.001190,-0.002518,-0.003113


In [58]:
eig_cols = [col for col in df1.columns if col.startswith("eig_")]
X1 = df1[eig_cols].values
y1 = df1["label"].values

In [37]:
# -----------------------------
# Fit PCA on the dataset
# -----------------------------
X_pca = pipe.fit_transform(X, pca_variance=256)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (5000, 256)
Num PCs kept: 256
Explained variance retained: 0.6685011753829773


In [38]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)

df_pca["label"] = y

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC248,PC249,PC250,PC251,PC252,PC253,PC254,PC255,PC256,label
0,21.083353,-14.120772,-17.793718,12.725747,5.943540,-3.104733,-17.236413,3.438575,7.126851,6.058824,...,1.574160,1.668983,-0.050340,3.314107,-1.061666,3.224879,2.500396,-1.108639,0.123194,0
1,18.069204,-8.402765,-25.374033,5.061179,11.622710,1.824561,-3.846996,13.787729,6.933621,-4.699521,...,-3.132098,-1.517956,2.169762,2.307395,5.784931,1.286442,-0.567060,2.987098,-0.558433,0
2,-28.805356,29.788096,36.715717,12.281802,-1.804486,16.762775,11.311653,8.079418,13.748440,8.856247,...,-2.615389,1.642098,-0.076667,-0.133228,1.614358,2.767360,-0.146837,-1.725690,1.529670,1
3,22.362664,-6.602939,-7.995164,6.662083,16.567196,0.361286,-15.740513,2.263746,-10.191050,-4.761345,...,0.653201,-2.047674,-1.455638,-2.023253,-1.674039,0.297322,1.779827,-1.452911,-0.779316,0
4,-14.360310,3.480982,23.497105,-5.537583,-3.984881,-12.366828,1.533904,5.112579,4.952311,-2.384536,...,-0.060616,0.945242,2.652566,0.820567,-1.457182,-0.273697,1.384398,-2.224756,3.323069,0


In [39]:
pd.DataFrame.to_csv(df_pca, r"../data/spectral/temp_1/triviaqa_pca.csv", index=False) 

## MMLU Data

In [ ]:
def build_training_dataset_eigen_only(jsonl_path: str, pt_path: str):

    with open(jsonl_path, "r") as f:
        df_meta = pd.DataFrame([json.loads(line) for line in f])

    pt_payload = torch.load(pt_path, map_location="cpu", weights_only=True)
    feature_dict = pt_payload.get("data", pt_payload)

    # Extract the eigenvalue arrays
    feature_rows = []
    for record_id, payload in list(feature_dict.items()):
        eig_array = payload['laplacian'].numpy().astype(np.float32)

        feature_rows.append({"id": record_id, "features": eig_array})

    df_features = pd.DataFrame(feature_rows)

    # Inner join to guarantee perfect alignment between features and labels
    df_final = pd.merge(df_meta, df_features, on="id", how="inner")

    # Filter errors and construct the binary label vector
    df_final = df_final[df_final["correctness"] != "error"]
    df_final["label"] = df_final["correctness"].apply(lambda x: 1 if x.lower() == "incorrect" else 0)

    # Formulate the X and y matrices
    X = np.vstack(df_final["features"].values)
    y = df_final["label"].values

    print(f"Feature Matrix (X) shape: {X.shape}")
    print(f"Label Vector (y) shape: {y.shape}")

    return df_final, X, y

In [60]:
jsonl_pt = r"../data/spectral/temp_1/mmlu_t1.0_n1.jsonl"
eigen_pt = r"../data/spectral/temp_1/mmlu_t1.0_n1_eigen.pt"

df2, X2, y2 = build_training_dataset_eigen_only(jsonl_pt, eigen_pt)

Feature Matrix (X) shape: (5000, 6720)
Label Vector (y) shape: (5000,)


In [42]:
# -----------------------------
# Fit PCA on the dataset
# -----------------------------
X_pca = pipe.fit_transform(X, pca_variance=256)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (5000, 256)
Num PCs kept: 256
Explained variance retained: 0.6462055


In [43]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)

df_pca["label"] = y

pd.DataFrame.to_csv(df_pca, r"../data/spectral/temp_1/mmlu_pca.csv", index=False) 

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC248,PC249,PC250,PC251,PC252,PC253,PC254,PC255,PC256,label
0,28.095181,19.478035,4.851603,-18.490540,-6.517684,-16.811602,-7.449831,-0.066988,10.273497,-7.211279,...,2.128880,2.454807,2.247278,0.259743,0.526306,-4.192413,3.461673,3.023068,-2.246560,0
1,17.192484,4.522415,5.463432,9.894576,-8.939204,-14.678575,-7.786435,9.399713,-8.389544,8.575562,...,-0.455081,1.094299,0.230971,1.700471,-0.291576,-1.179935,-2.491132,-4.070527,-0.993974,0
2,25.700346,-15.575506,7.821413,1.181440,-15.325275,-12.097673,15.802523,6.666360,-6.045353,-3.426772,...,-2.400367,-1.224737,-0.654475,-3.543831,2.308105,2.421879,1.647243,1.746159,0.008811,1
3,-32.283199,29.817335,-12.415548,12.504129,-14.984789,5.352255,2.917861,13.761698,-9.107186,-3.771079,...,3.105021,-0.504867,0.755770,0.062147,0.989702,-0.261222,0.643981,0.111251,-1.529331,1
4,8.332973,-12.712507,-7.955024,3.518253,-0.781295,5.736643,3.065584,-2.108086,7.123981,1.174061,...,-0.930953,-0.133053,0.419242,-1.843885,-2.172737,3.124592,0.099813,-3.596759,-0.276429,0


## MMLU + Triviaqa

In [85]:
X = np.vstack([X1, X2])
y = np.hstack([y1, y2])

X.shape, y.shape

((10000, 6720), (10000,))

In [86]:
# -----------------------------
# Fit PCA on the dataset
# -----------------------------
X_pca = pipe.fit_transform(X, pca_variance=384)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (10000, 384)
Num PCs kept: 384
Explained variance retained: 0.7133489784094807


In [87]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)

df_pca["label"] = y

pd.DataFrame.to_csv(df_pca, r"../data/spectral/temp_1/triviaqa_mmlu_pca.csv", index=False) 

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC376,PC377,PC378,PC379,PC380,PC381,PC382,PC383,PC384,label
0,-1.094865,27.862167,-26.793908,12.981636,9.381119,6.708258,-1.340294,-5.389115,-6.179338,13.179383,...,1.141642,-1.513671,-0.356864,-2.466488,0.522463,-0.797013,-2.336710,1.125533,2.807862,0
1,-2.032936,14.817370,-23.542576,16.847013,14.676365,-1.994764,6.884055,-6.623255,-10.491322,-4.347425,...,0.640012,1.805900,-0.805777,0.104316,1.369356,-1.792999,-2.741111,4.162301,-2.164264,0
2,39.350086,10.462175,47.182297,-3.746812,0.416630,3.043195,2.298111,13.113747,-4.731201,3.543718,...,1.910907,-0.814500,-0.884993,2.854972,-1.212797,-1.341396,-0.516281,1.186963,3.045816,1
3,-5.219615,24.006729,-16.933551,8.035086,-2.114434,8.607027,7.773032,-5.978483,-8.853668,-0.382581,...,-0.982953,3.229909,1.547978,-1.311842,-1.556801,-4.663886,-1.705123,1.689088,-1.730384,0
4,31.986464,16.919031,12.676443,-7.486537,-5.267607,-6.135123,-12.971660,1.598459,4.758632,-8.732115,...,-0.810313,0.893578,-3.911738,-0.574768,-0.780606,1.841879,1.040061,1.211693,1.978020,0


## Halueval Data

In [7]:
def build_training_dataset_eigen_only(jsonl_path: str, pt_path: str):

    with open(jsonl_path, "r") as f:
        df_meta = pd.DataFrame([json.loads(line) for line in f])

    pt_payload = torch.load(pt_path, map_location="cpu", weights_only=True)
    feature_dict = pt_payload.get("data", pt_payload)

    # Extract the eigenvalue arrays
    feature_rows = []
    for record_id, payload in list(feature_dict.items()):
        eig_array = payload['laplacian'].numpy().astype(np.float32)

        feature_rows.append({"id": record_id, "features": eig_array})

    df_features = pd.DataFrame(feature_rows)

    # Inner join to guarantee perfect alignment between features and labels
    df_final = pd.merge(df_meta, df_features, on="id", how="inner")

    # Filter errors and construct the binary label vector
    df_final = df_final[df_final["correctness"] != "error"]
    df_final["label"] = df_final["correctness"].apply(lambda x: 1 if x.lower() == "incorrect" else 0)

    # Formulate the X and y matrices
    X = np.vstack(df_final["features"].values)
    y = df_final["label"].values

    print(f"Feature Matrix (X) shape: {X.shape}")
    print(f"Label Vector (y) shape: {y.shape}")

    return df_final, X, y

In [8]:
jsonl_pt = r"../data/spectral/temp_1/halueval_t1.5_n1_1000s_prompt2.jsonl"
eigen_pt = r"../data/spectral/temp_1/halueval_t1.5_n1_eigen_1000s_prompt2.pt"

df, X, y = build_training_dataset_eigen_only(jsonl_pt, eigen_pt)

Feature Matrix (X) shape: (10000, 6720)
Label Vector (y) shape: (10000,)


In [11]:
# -----------------------------
# Fit PCA on the dataset
# -----------------------------
X_pca = pipe.fit_transform(X, pca_variance=256)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (10000, 256)
Num PCs kept: 256
Explained variance retained: 0.6068003


In [12]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)

df_pca["label"] = y

pd.DataFrame.to_csv(df_pca, r"../data/spectral/temp_1/halueval_t1.0_n1_pca.csv", index=False) 

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC248,PC249,PC250,PC251,PC252,PC253,PC254,PC255,PC256,label
0,23.745983,-4.078446,-5.536303,-9.591763,3.527650,-5.588723,9.179587,4.338895,-7.959638,-4.226568,...,0.837641,-1.462271,1.216666,3.700047,-2.164570,-1.453887,-0.286320,2.479606,-0.426116,1
1,-8.324484,-3.371192,16.541830,2.547776,-7.275225,24.000263,5.934997,6.335870,-16.427061,1.983333,...,-1.390460,-0.247518,1.140044,-3.327961,-1.614570,-1.825011,1.718269,0.958246,0.257802,0
2,-35.974052,-7.751545,-39.317482,19.957874,1.220803,-24.410509,-0.044090,9.653060,-9.045791,4.235573,...,-1.851807,-0.317285,-2.679782,2.727677,2.070567,0.273773,-0.468052,-1.023675,1.443898,0
3,15.101802,-7.308723,4.300932,-6.253857,1.836298,4.708517,-3.641397,9.053051,0.477310,-7.454600,...,3.402055,1.062985,-1.837196,-2.997785,4.333920,-0.594758,-0.880039,-2.122211,-2.418713,0
4,-11.150685,0.132028,16.755690,-13.999758,-8.046125,10.271860,-7.938184,6.917554,3.589669,5.085630,...,-3.978410,1.816745,-0.400447,0.926974,-2.452971,-0.977409,-0.835276,-1.127688,-0.502392,1


## nq_Open Data

In [3]:
def build_training_dataset_eigen_only(jsonl_path: str, pt_path: str):

    with open(jsonl_path, "r") as f:
        df_meta = pd.DataFrame([json.loads(line) for line in f])

    pt_payload = torch.load(pt_path, map_location="cpu", weights_only=True)
    feature_dict = pt_payload.get("data", pt_payload)

    # Extract the eigenvalue arrays
    feature_rows = []
    for record_id, payload in list(feature_dict.items()):
        eig_array = payload['laplacian'].numpy().astype(np.float32)

        feature_rows.append({"id": record_id, "features": eig_array})

    df_features = pd.DataFrame(feature_rows)

    # Inner join to guarantee perfect alignment between features and labels
    df_final = pd.merge(df_meta, df_features, on="id", how="inner")

    # Filter errors and construct the binary label vector
    df_final = df_final[df_final["correctness"] != "error"]
    df_final["label"] = df_final["correctness"].apply(lambda x: 1 if x.lower() == "correct" else 0)

    # Formulate the X and y matrices
    X = np.vstack(df_final["features"].values)
    y = df_final["label"].values

    print(f"Feature Matrix (X) shape: {X.shape}")
    print(f"Label Vector (y) shape: {y.shape}")

    return df_final, X, y

In [4]:
jsonl_pt = r"../data/spectral/temp_1/nq_open_t1.0_n1_3600s.jsonl"
eigen_pt = r"../data/spectral/temp_1/nq_open_t1.0_n1_eigen_3600s.pt"

df, X, y = build_training_dataset_eigen_only(jsonl_pt, eigen_pt)

Feature Matrix (X) shape: (3600, 6720)
Label Vector (y) shape: (3600,)


In [6]:
# -----------------------------
# Fit PCA on the dataset
# -----------------------------
pca_variance = 128
X_pca = pipe.fit_transform(X, pca_variance)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

PCA output shape: (3600, 128)
Num PCs kept: 128
Explained variance retained: 0.5867524


In [7]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)

df_pca["label"] = y

pd.DataFrame.to_csv(df_pca, r"../data/spectral/temp_1/nq_open.csv", index=False) 

df_pca.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC120,PC121,PC122,PC123,PC124,PC125,PC126,PC127,PC128,label
0,-28.606779,-10.033399,-0.936449,20.720320,-10.276968,-2.741010,3.564272,-4.955451,-0.203194,6.748618,...,-1.870915,-2.801402,4.684344,-2.188635,4.586585,-1.574654,-3.917422,2.485690,-0.186193,1
1,6.530851,8.132322,-22.158134,-16.959623,-7.864002,-6.322720,-16.405203,-1.852743,7.896513,4.429062,...,-0.956909,-1.230122,-0.714431,-3.017431,2.063331,0.773087,-1.687441,-0.564145,0.879404,0
2,-13.539382,-12.022514,-11.374075,7.810328,0.082710,-8.850789,1.351140,-9.890763,13.091425,-9.432151,...,0.309495,-0.734851,-1.454404,0.542696,-0.253970,0.535085,-1.842551,-0.497847,-3.026887,0
3,-14.507421,1.560368,3.363966,-1.508545,13.914618,-13.189336,-6.720327,-6.447372,-1.174421,-5.769156,...,4.535730,-4.748932,4.168555,1.210364,-1.349309,1.865327,-2.370028,2.902025,-4.780513,0
4,-1.169939,11.841800,11.852506,-8.863430,-9.387660,-14.981409,-16.684906,2.827239,2.943560,6.609247,...,-0.022526,0.187337,0.201103,-2.521518,-2.058326,1.712582,4.828406,-2.326826,1.851398,0


## We can now merge all the datasets and do PCA on the whole data.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# -----------------------------
#  List the 4 .pt files
# -----------------------------
pt_paths = [
    r"../data/spectral/temp_1/triviaqa_Judgelabels_and_eigs_top10.pt",
    # r"../data/spectral/mmlu_Judgelabels_and_eigs_top10.pt",
    # r"../data/spectral/halueval_Judgelabels_and_eigs_top10.pt"
]

# Optional: dataset names for tracking
dataset_names = ["triviaqa"]

# -----------------------------
# Load + featurize all four, then stack
# -----------------------------
X_all = []
ids_all = []
source_all = []

for path, ds_name in zip(pt_paths, dataset_names):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file: {path}")

    X, ids = pipe.load_pt(path, feature_key="eig_top10")  # X shape: [N, L*K] after head-avg + signed-log
    X_all.append(X)
    ids_all.extend(ids)
    source_all.extend([ds_name] * len(ids))

X_all = np.vstack(X_all)  # shape: [N_total, L*K]
print("Combined X shape:", X_all.shape)

In [ ]:
# -----------------------------
# 3) Fit PCA on the combined dataset
# -----------------------------
X_pca = pipe.fit_transform(X_all)
print("PCA output shape:", X_pca.shape)
print("Num PCs kept:", pipe.pca.n_components_)
print("Explained variance retained:", pipe.pca.explained_variance_ratio_.sum())

In [ ]:
# -----------------------------
# 4) Put into a DataFrame for convenience
# -----------------------------
pc_cols = [f"PC{i+1}" for i in range(X_pca.shape[1])]
df_pca = pd.DataFrame(X_pca, columns=pc_cols)
df_pca.insert(0, "id", ids_all)
df_pca.insert(1, "dataset", source_all)

df_pca.head()

In [ ]:
import torch

torch.save(
    {
        "X": torch.tensor(
            df_pca.filter(like="PC").values,
            dtype=torch.float32
        ),
        "id": df_pca["id"].tolist(),
        "dataset": df_pca["dataset"].tolist()
    },
    "../data/processed/df_pca.pt"
)

In [ ]:
df_pca.head()